# Model selection — inspecting the bake-off by hand

This is a **stub**: it loads what `run_pipeline.cmd bench` wrote and gives you the
handful of views that are hard to get from the Markdown report. Extend it freely —
nothing downstream reads this notebook.

The winner is picked by **one F score over the whole goldstandard** (highest wins).
That is a single number, so it hides a lot; these tables exist so you can check it
before trusting it — especially *which categories* a model gets wrong.

Inputs, all under `results/model_selection/`:

| file | one row per |
|---|---|
| `model_scores.csv` | (model, metric) — long format |
| `category_scores.csv` | (model, dimension, category) — tp/fp/fn, P/R/F1, both supports |
| `paper_comparison.csv` | (model, paper, dimension) — human value vs model value |
| `predictions_<model>.csv` | (model, paper) — the raw annotation, incl. `llm_error` |

Run `run_pipeline.cmd bench "" "" score` to regenerate them from predictions already
on disk (free, no API calls).

In [ ]:
import json, os
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

# Same data root the pipeline uses: LNI_DATA_ROOT if set, else the study folder.
STUDY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ROOT = Path(os.environ.get("LNI_DATA_ROOT") or STUDY)
SEL = ROOT / "results" / "model_selection"
print(SEL, "exists:", SEL.exists())
sorted(p.name for p in SEL.glob("*")) if SEL.exists() else []

In [ ]:
selection = json.loads((SEL / "model_selection.json").read_text(encoding="utf-8"))
scores = pd.read_csv(SEL / "model_scores.csv")
cats = pd.read_csv(SEL / "category_scores.csv")
comp = pd.read_csv(SEL / "paper_comparison.csv")

WINNER = (selection.get("winner") or {}).get("model")
print("winner :", WINNER)
print("why    :", selection.get("winner_reason"))
print("coder  :", selection.get("gold_coder"), "|", selection.get("n_papers"),
      "papers,", selection.get("n_accepted"), "accepted as RS")
print("models :", sorted(scores["model"].unique()))

## 1. The ranking

`overall` = `gate_weight × gate macro-F1 + (1 − gate_weight) × mean(dimension scores)`.
A model below the coverage floor is disqualified no matter how it scored — check
`coverage` before reading anything else.

In [ ]:
wide = scores.pivot(index="model", columns="metric", values="value")
cols = [c for c in ["coverage", "overall", "typology", "gate_macro_f1", "gate_accuracy",
                    "gate_f1_rs", "gate_f1_not_rs"] if c in wide.columns]
wide[cols].sort_values("overall", ascending=False).round(3)

## 2. Per dimension

Where the models actually differ. `research_position` is exact match (single-valued);
the other four are micro-F1 over the label sets.

In [ ]:
# `dim:<d>` is the dimension score itself; `dim:<d>:precision` etc. are its parts.
dims = scores[scores["metric"].str.count(":") == 1].copy()
dims["dimension"] = dims["metric"].str.split(":").str[1]
dims.pivot(index="model", columns="dimension", values="value").round(3)

## 3. Per category — where the score comes from

The dimension score averages over categories, so a model that **never predicts a rare
key** looks fine there. This is the table that shows it: recall 0 at a high
`support_human`. A large `support_model` vs `support_human` gap in the other direction
is over-application (the model tags everything with that key).

`in_schema == False` means the model invented a key that is not in
`category_schema.yaml`. Those count as false positives — if one is a genuine synonym,
add it to that category's `examples:` rather than leave the model punished for it.

In [ ]:
MODEL = WINNER  # <- set to any id in scores["model"] to inspect a different candidate

c = cats[(cats["model"] == MODEL) & (cats["support_human"] > 0)]
c.sort_values(["f1", "support_human"], ascending=[True, False])[
    ["dimension", "category", "f1", "precision", "recall",
     "support_human", "support_model", "tp", "fp", "fn", "in_schema"]
].head(30)

In [ ]:
# Keys the model produced that the humans never used (over-application / invention).
cats[(cats["model"] == MODEL) & (cats["support_human"] == 0) & (cats["support_model"] > 0)][
    ["dimension", "category", "support_model", "in_schema"]
].sort_values("support_model", ascending=False)

## 4. Drill into one category

Pick a row from the table above, then read the papers behind it. `missed` are keys the
human used and the model did not; `extra` are the model's additions.

In [ ]:
DIMENSION = "software_type"   # <- any of cats["dimension"].unique()
CATEGORY = None                # <- e.g. "simulation"; None = the whole dimension

d = comp[(comp["model"] == MODEL) & (comp["dimension"] == DIMENSION)]
if CATEGORY:
    hit = lambda s: d[s].fillna("").str.contains(rf"\b{CATEGORY}\b", regex=True)
    d = d[hit("human") | hit("model_value")]
print(d["status"].value_counts().to_dict())
d.sort_values("jaccard")[["id", "title", "human", "model_value", "status",
                          "missed", "extra", "jaccard"]].head(25)

## 5. Papers the models disagree about

If every candidate misses the same paper, the problem is usually the prompt, the schema
or the human coding — not the model. Those are worth reading before changing anything.

In [ ]:
agree = (comp.assign(ok=comp["status"].eq("agree"))
             .groupby(["dimension", "id"], dropna=False)["ok"]
             .agg(["sum", "count"])
             .rename(columns={"sum": "models_agreeing", "count": "models_scored"}))
hard = agree[agree["models_agreeing"] == 0].reset_index()
titles = comp.drop_duplicates("id").set_index("id")["title"]
hard["title"] = hard["id"].map(titles)
print(len(hard), "(dimension, paper) pairs that NO model got right")
hard.sort_values(["dimension", "id"]).head(25)

In [ ]:
# The gate on its own: it carries half the overall score.
g = comp[comp["dimension"] == "label_research_software"]
pd.crosstab([g["model"], g["human"]], g["model_value"])

## 6. Raw answers

When a cell above looks wrong, the raw response is in `predictions_<model>.csv` —
including `llm_error` for the papers that were excluded from the scores entirely.

In [ ]:
slug = (MODEL or "").replace("/", "_").replace(":", "_")
path = next(iter(SEL.glob(f"predictions_*{slug}*.csv")), None)
raw = pd.read_csv(path) if path else pd.DataFrame()
print(path)
raw[raw.get("llm_error").notna()] if "llm_error" in raw else raw.head()